In [ ]:
# Example 1: Serial class construction (no port -> not opened, safe in CI).
import serial

s = serial.Serial()
s.baudrate = 9600
s.bytesize = serial.EIGHTBITS
s.parity = serial.PARITY_NONE
s.stopbits = serial.STOPBITS_ONE
s.timeout = 1

print(f"baudrate = {s.baudrate}")
print(f"bytesize = {s.bytesize}")
print(f"parity   = {s.parity}")
print(f"stopbits = {s.stopbits}")
print(f"timeout  = {s.timeout}")
print(f"is_open  = {s.is_open}")

assert s.baudrate == 9600
assert s.is_open is False

In [ ]:
# Example 2: list available serial ports via serial.tools.list_ports.
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
print(f"Found {len(ports)} serial port(s)")
for p in ports[:10]:
    print(f"  device={p.device}  desc={p.description}  hwid={p.hwid}")

# In containers / CI there are typically zero ports; that's still a valid result.
assert isinstance(ports, list)

In [ ]:
# Example 3: serial.threaded helpers exist and are importable.
import serial.threaded

class EchoProtocol(serial.threaded.Protocol):
    def __init__(self):
        self.received = bytearray()

    def data_received(self, data):
        self.received.extend(data)

proto = EchoProtocol()
proto.data_received(b"hello ")
proto.data_received(b"world")

assert bytes(proto.received) == b"hello world"
print(f"received bytes: {bytes(proto.received)!r}")

In [ ]:
# Example 4: URL handler (loop://) - in-memory loopback, no real hardware.
import serial

s = serial.serial_for_url("loop://", timeout=1)
try:
    payload = b"pyserial-smoke-test\n"
    n = s.write(payload)
    assert n == len(payload)
    echoed = s.read(len(payload))
    print(f"wrote   = {payload!r}")
    print(f"echoed  = {echoed!r}")
    assert echoed == payload
finally:
    s.close()

assert s.is_open is False

In [ ]:
# Example 5: rs485 settings + SerialException is exposed at top level.
import serial
import serial.rs485

settings = serial.rs485.RS485Settings(
    rts_level_for_tx=True,
    rts_level_for_rx=False,
    delay_before_tx=0.001,
    delay_before_rx=0.001,
)
print(f"rts_for_tx       = {settings.rts_level_for_tx}")
print(f"delay_before_tx  = {settings.delay_before_tx}")

# Verify the canonical exception class is importable from the top-level module.
exc_cls = serial.SerialException
assert issubclass(exc_cls, Exception)

# Opening a clearly-bogus port must raise SerialException (and not a generic OSError).
try:
    serial.Serial("/definitely/not/a/real/port/xyz", timeout=0.1)
except serial.SerialException as e:
    print(f"got expected SerialException: {type(e).__name__}")

assert settings.rts_level_for_tx is True